# Module 1: Why OpenEnv? — Your First Environments

In this notebook you'll connect to three real hosted OpenEnv environments and interact with each using the same 3-method interface: `reset()`, `step()`, `state()`.

**Time:** ~15 min · **Difficulty:** Beginner · **GPU:** Not required

In [1]:
# Install dependencies
!pip install -q openenv-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.1/174.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.8/633.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.1 MB/s eta 0:00:00


## 1. The Echo Environment

The simplest possible OpenEnv environment — it echoes back whatever you send. Perfect for learning the interface.

Hosted at: `https://openenv-echo-env.hf.space`

In [ ]:
from envs.echo_env import EchoEnv, EchoAction

# Connect to the hosted Echo environment
with EchoEnv(base_url="https://openenv-echo-env.hf.space").sync() as env:
    # reset() starts a new episode
    result = env.reset()
    print("After reset:")
    print(f"  Observation: {result.observation}")
    print(f"  Done: {result.done}")
    print()

    # step() takes an action and returns the next observation
    result = env.step(EchoAction(message="Hello, OpenEnv!"))
    print("After step:")
    print(f"  Observation: {result.observation}")
    print(f"  Reward: {result.reward}")
    print(f"  Done: {result.done}")
    print()

    # state() returns episode metadata
    state = env.state()
    print("State:")
    print(f"  Episode ID: {state.episode_id}")
    print(f"  Step count: {state.step_count}")

Three methods. That's the entire API. Every OpenEnv environment works exactly like this.

## 2. OpenSpiel Catch

Now let's connect to a real game. Catch is a simple single-player game from DeepMind's OpenSpiel:

- A ball falls from the top of a 10×5 grid
- You move a paddle left/right to catch it
- Actions: `0` = left, `1` = stay, `2` = right
- Reward: `+1` if caught, `0` if missed

Same 3 methods, completely different game.

In [ ]:
!pip install -q git+https://huggingface.co/spaces/openenv/openspiel-env

In [ ]:
from envs.openspiel_env import OpenSpielEnv
from envs.openspiel_env.models import OpenSpielAction

OPENSPIEL_URL = "https://openenv-openspiel-catch.hf.space"

with OpenSpielEnv(base_url=OPENSPIEL_URL).sync() as env:
    result = env.reset()
    print(f"Game: Catch")
    print(f"Legal actions: {result.observation.legal_actions}")
    print(f"Info state length: {len(result.observation.info_state)}")
    print()

    # Play a few steps with a random policy
    import random
    step = 0
    while not result.done:
        action_id = random.choice(result.observation.legal_actions)
        action_name = {0: "LEFT", 1: "STAY", 2: "RIGHT"}[action_id]
        result = env.step(OpenSpielAction(
            action_id=action_id,
            game_name="catch"
        ))
        step += 1
        print(f"Step {step}: {action_name} → reward={result.reward}, done={result.done}")

    print(f"\nFinal reward: {result.reward}")
    print(f"State: {env.state()}")

Same pattern: `reset()` → `step()` → check `done`. The observation type is different (`OpenSpielObservation` vs `EchoObservation`), but the interface is identical.

## 3. TextArena Wordle

TextArena is a text-based game environment. Wordle gives you 6 attempts to guess a 5-letter word, with color-coded feedback after each guess.

Hosted at: `https://burtenshaw-textarena.hf.space`

In [ ]:
!pip install -q git+https://huggingface.co/spaces/burtenshaw/textarena

In [ ]:
from envs.textarena_env import TextArenaEnv, TextArenaAction

TEXTARENA_URL = "https://burtenshaw-textarena.hf.space"

with TextArenaEnv(base_url=TEXTARENA_URL).sync() as env:
    result = env.reset()
    print("Wordle prompt:")
    print(result.observation.prompt)
    print()

    # Make a guess
    guesses = ["crane", "slate", "blind"]
    for guess in guesses:
        if result.done:
            break
        result = env.step(TextArenaAction(message=f"[{guess}]"))
        print(f"Guess: {guess}")
        for msg in result.observation.messages:
            print(f"  [{msg.category}] {msg.content}")
        print(f"  Reward: {result.reward}, Done: {result.done}")
        print()

## 4. Async vs Sync

OpenEnv clients are async by default. For notebooks and simple scripts, use the `.sync()` wrapper:

```python
# Sync (notebooks, simple scripts)
with EchoEnv(base_url=url).sync() as env:
    result = env.reset()

# Async (production, training loops)
async with EchoEnv(base_url=url) as env:
    result = await env.reset()
```

For this course, we'll use `.sync()` everywhere for simplicity.

## Summary

You connected to three completely different environments — Echo, Catch, Wordle — using the same interface:

| Method | What it does |
|--------|--------------|
| `reset()` | Start a new episode |
| `step(action)` | Take an action, get observation + reward |
| `state()` | Get episode metadata |

The action and observation types change per environment, but the pattern never does.

**Next:** [Module 2](../module-2/README.md) — Using existing environments to build and compare policies.